In [4]:
import os
import math
import torch
import torch.nn as nn
import torch.optim as optim
import random
import glob
import numpy as np
from torch.utils.data import ConcatDataset
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
def set_seed(seed=42):
    """Fissa tutti i generatori di numeri casuali per la riproducibilità"""
    random.seed(seed)                      # Fissa il seed di Python base
    np.random.seed(seed)                   # Fissa il seed di Numpy
    torch.manual_seed(seed)                # Fissa il seed di PyTorch (CPU)
    torch.cuda.manual_seed(seed)           # Fissa il seed di PyTorch (GPU)
    torch.cuda.manual_seed_all(seed)       # Fissa il seed se hai più GPU


class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=2, batch_first=True)
        self.ln = nn.LayerNorm(channels)
        self.ff_self = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        size = x.shape[-1]
        x_flat = x.reshape(-1, self.channels, size * size).transpose(1, 2)
        x_ln = self.ln(x_flat)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x_flat
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.transpose(1, 2).reshape(-1, self.channels, size, size)

class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=32, num_groups=4):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.act1 = nn.SiLU()

        self.time_mlp = nn.Linear(time_dim, out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act2 = nn.SiLU()

        self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act1(self.gn1(self.conv1(x)))
        time_proj = self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + time_proj
        h = self.act2(self.gn2(self.conv2(h)))
        return h + self.residual_conv(x)

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class ImprovedUNet(nn.Module):
    def __init__(self, in_channels=3, base_channels=64):
        super().__init__()

        time_dim = base_channels * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim)
        )

        self.inc = UNetBlock(in_channels, base_channels, time_dim)

        self.down1 = nn.Conv2d(base_channels, base_channels, 4, 2, 1)
        self.enc1 = UNetBlock(base_channels, base_channels * 2, time_dim)

        self.down2 = nn.Conv2d(base_channels*2, base_channels*2, 4, 2, 1)
        self.enc2 = UNetBlock(base_channels * 2, base_channels * 4, time_dim)

        self.down3 = nn.Conv2d(base_channels*4, base_channels*4, 4, 2, 1)

        self.bottleneck1 = UNetBlock(base_channels * 4, base_channels * 8, time_dim)
        self.attention = SelfAttention(base_channels * 8)
        self.bottleneck2 = UNetBlock(base_channels * 8, base_channels * 4, time_dim)

        self.up1 = nn.ConvTranspose2d(base_channels * 4, base_channels * 4, kernel_size=2, stride=2)
        self.dec1 = UNetBlock(base_channels * 8, base_channels * 2, time_dim)

        self.up2 = nn.ConvTranspose2d(base_channels * 2, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = UNetBlock(base_channels * 4, base_channels, time_dim)

        self.up3 = nn.ConvTranspose2d(base_channels, base_channels, kernel_size=2, stride=2)
        self.dec3 = UNetBlock(base_channels * 2, base_channels, time_dim)

        self.out = nn.Conv2d(base_channels, in_channels, kernel_size=3, padding=1)

    def forward(self, x_t, t):
        t_emb = self.time_mlp(t.float())

        s1 = self.inc(x_t, t_emb)
        h = self.down1(s1)
        s2 = self.enc1(h, t_emb)
        h = self.down2(s2)
        s3 = self.enc2(h, t_emb)
        h = self.down3(s3)

        h = self.bottleneck1(h, t_emb)
        h = self.attention(h)
        h = self.bottleneck2(h, t_emb)

        h = self.up1(h)
        h = torch.cat([h, s3], dim=1)
        h = self.dec1(h, t_emb)

        h = self.up2(h)
        h = torch.cat([h, s2], dim=1)
        h = self.dec2(h, t_emb)

        h = self.up3(h)
        h = torch.cat([h, s1], dim=1)
        h = self.dec3(h, t_emb)

        return self.out(h)

class DDPM(nn.Module):
    def __init__(self, network, n_steps=200, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.network = network
        self.n_steps = n_steps

        beta = torch.linspace(beta_start, beta_end, n_steps)
        alpha = 1.0 - beta
        alpha_cumprod = torch.cumprod(alpha, dim=0)
        alpha_cumprod_prev = torch.cat(
            [torch.tensor([1.0]), alpha_cumprod[:-1]]
        )

        self.register_buffer('alpha_cumprod_prev', alpha_cumprod_prev)
        self.register_buffer('posterior_variance', beta * (1 - alpha_cumprod_prev) / (1 - alpha_cumprod))
        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('alpha_cumprod', alpha_cumprod)
        self.register_buffer('sqrt_alpha_cumprod', torch.sqrt(alpha_cumprod))
        self.register_buffer('sqrt_one_minus_alpha_cumprod', torch.sqrt(1.0 - alpha_cumprod))

    def forward_diffusion(self, x_0, t, noise):
        sqrt_alpha_cumprod = self.sqrt_alpha_cumprod[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_cumprod = self.sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1)
        return sqrt_alpha_cumprod * x_0 + sqrt_one_minus_alpha_cumprod * noise

    def compute_loss(self, x_0):
        batch_size = x_0.shape[0]
        t = torch.randint(0, self.n_steps, (batch_size,), device=x_0.device)
        noise = torch.randn_like(x_0)

        x_t = self.forward_diffusion(x_0, t, noise)
        predicted_noise = self.network(x_t, t.float())

        return nn.MSELoss()(predicted_noise, noise)

    @torch.no_grad()
    def sample(self, batch_size, channels, height, width):
        self.network.eval()
        device = self.beta.device

        x = torch.randn(batch_size, channels, height, width, device=device)

        for i in reversed(range(self.n_steps)):
            t = torch.full((batch_size,), i, dtype=torch.long, device=device)
            predicted_noise = self.network(x, t.float())

            beta_t = self.beta[i]
            alpha_t = self.alpha[i]
            alpha_cumprod_t = self.alpha_cumprod[i]

            mean = (1 / torch.sqrt(alpha_t)) * (x - ((beta_t / torch.sqrt(1 - alpha_cumprod_t)) * predicted_noise))

            if i > 0:
                noise = torch.randn_like(x)
                sigma_t = torch.sqrt(self.posterior_variance[i])
                x = mean + sigma_t * noise
            else:
                x = mean

        self.network.train()
        return x

if __name__ == "__main__":

    set_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Dispositivo in uso: {device}")

    torch.backends.cudnn.benchmark = True

    output_dir = "/content/drive/MyDrive/Risultati_DDPM"
    os.makedirs(output_dir, exist_ok=True)

    ####### PREPARAZIONE DATI

    transform_train = transforms.Compose([
        transforms.Resize(72),
        transforms.RandomCrop(64),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    print("Caricamento del dataset...")

    dataset_train = datasets.Flowers102(root='./data', split='train', download=True, transform=transform_train)
    dataset_val = datasets.Flowers102(root='./data', split='val', download=True, transform=transform_train)
    dataset_test = datasets.Flowers102(root='./data', split='test', download=True, transform=transform_train)

    full_train_dataset = ConcatDataset([dataset_train, dataset_val, dataset_test]) # uniamo per avere più immagini per il training
    train_dataloader = DataLoader(full_train_dataset, batch_size=64, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)

    ###### INIZIALIZZAZIONE MODELLO E OTTIMIZZATORE

    nn_architecture = ImprovedUNet(in_channels=3, base_channels=64).to(device)
    ddpm = DDPM(nn_architecture, n_steps=1000).to(device)

    optimizer = optim.AdamW(ddpm.parameters(), lr=2e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    scaler = torch.cuda.amp.GradScaler()
    epochs = 1000
    print(f"Inizio del training definitivo ({epochs} epoche)...")
    start_epoch = 0

    # Cerchiamo nella cartella drive se esistono addestramenti precedenti.
    # Se li trova, carica i pesi (load_state_dict) e riparte dall'ultima epoca salvata

    checkpoints = glob.glob(os.path.join(output_dir, "ddpm_epoch_*.pth"))

    if checkpoints:

        checkpoints.sort(key=lambda x: int(x.split('_')[-1].split('.')[0]))
        last_checkpoint = checkpoints[-1] # prendiamo il file più recente

        print(f"Trovato checkpoint precedente: {last_checkpoint}")

        # 1. Carichiamo l'intero dizionario del checkpoint
        checkpoint = torch.load(last_checkpoint, map_location=device)

        # 2. Ripristiniamo gli stati dei vari componenti
        ddpm.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])
        scheduler.load_state_dict(checkpoint["scheduler"])

        # 3. Estraiamo l'epoca direttamente dal checkpoint salvato
        start_epoch = checkpoint["epoch"] + 1

        print(f"Riprendo l'addestramento dall'epoca {start_epoch + 1}...")
    else:
        print("Nessun checkpoint trovato. Inizio l'addestramento da zero.")

    ###### LOOP DI TRAINING

    for epoch in range(start_epoch, epochs):
        ddpm.train()
        total_train_loss = 0

        pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1:03d}/{epochs}")
        for batch in pbar:
            x_0 = batch[0].to(device)
            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type='cuda'):
                loss = ddpm.compute_loss(x_0)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(ddpm.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            total_train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = total_train_loss / len(train_dataloader)
        print(f"Epoch {epoch+1:03d} completata | Loss Media: {avg_train_loss:.4f}")

        scheduler.step()

        ###### VISUALIZZAZIONE OGNI 50 EPOCHE

        if (epoch + 1) % 20 == 0:
            print(f"\n---> Generazione test all'epoca {epoch+1}...")

            # Salva i pesi
            torch.save({
                "epoch": epoch,
                "model": ddpm.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "scheduler": scheduler.state_dict()
            },os.path.join(output_dir, f"ddpm_epoch_{epoch+1}.pth"))

            # Generazione Test
            ddpm.eval()
            with torch.no_grad():
                generated_samples = ddpm.sample(batch_size=16, channels=3, height=64, width=64)

            gen_vis = torch.clamp((generated_samples.cpu() + 1.0) / 2.0, 0.0, 1.0)
            grid_generated = make_grid(gen_vis, nrow=4).permute(1, 2, 0).numpy()

            plt.figure(figsize=(6, 6))
            plt.imshow(grid_generated)
            plt.title(f"Immagini Generate - Epoca {epoch+1}")
            plt.axis('off')
            plt.show()

    print("\nTraining Completato con Successo!")

Output hidden; open in https://colab.research.google.com to view.